In [ ]:
import pandas as pd
from PIL import Image
from tqdm import tqdm
import openslide
import numpy as np
from glob import glob
from os.path import join as opj
import os

In [ ]:
def alignment_filter_bak(x):
    #if "FS" in x["Block"]: return False
        
    block_stains = set(x["Stain"])
    if "H&E" not in block_stains: return False
    if not block_stains.difference({"H&E"}): return False
    return True

def alignment_filter(x):
    if ("FS" in x["Block"]) and (x["Block"].endswith("x")):
        return False
    
    block_stains = set(x["Stain"])

    if "H&E" not in block_stains:
        return False

    return True


In [ ]:
def get_thumbnail(slide, target_downsample=64):
    tn_level = 2
    tn_downsample = slide.level_downsamples[tn_level]
    thumb = slide.read_region(location=(0,0), level = tn_level, size=slide.level_dimensions[tn_level])
    if not (int(tn_downsample) == target_downsample):
        scale_factor = tn_downsample/target_downsample
        original_width, original_height = thumb.size
        new_width = int(original_width * scale_factor)
        new_height = int(original_height * scale_factor)
        thumb = thumb.resize((new_width, new_height))
    return np.array(thumb.convert('RGB'))


In [ ]:
# def resize_and_pad_to_size(images, target_size):
#     """
#     Resizes images that are too large to fit within the target size, 
#     and pads smaller images to the target size, aligning to the top-left corner.
#     
#     Args:
#         images (list of PIL.Image.Image): List of images to process.
#         target_size (tuple): Target size as (width, height).
#     
#     Returns:
#         list of PIL.Image.Image: List of processed images.
#     """
#     target_width, target_height = target_size
#     processed_images = []
#     
#     for im in images:
#         # Resize the image if it's larger than the target size
#         if im.width > target_width or im.height > target_height:
#             im.thumbnail((target_width, target_height), Image.LANCZOS)
#         
#         # Create a blank canvas with the target size
#         canvas = Image.new("RGB", target_size, color=(0, 0, 0))
#         
#         # Paste the resized image onto the canvas at the top-left
#         canvas.paste(im, (0, 0))
#         
#         processed_images.append(canvas)
#     
#     return processed_images

In [ ]:
def resize_and_pad_to_size_one_im(im, target_size):
    """
    Resizes images that are too large to fit within the target size, 
    and pads smaller images to the target size, aligning to the top-left corner.
    
    Args:
        images (list of PIL.Image.Image): List of images to process.
        target_size (tuple): Target size as (width, height).
    
    Returns:
        list of PIL.Image.Image: List of processed images.
    """
    target_width, target_height = target_size

    # Resize the image if it's larger than the target size
    if im.width > target_width or im.height > target_height:
        im.thumbnail((target_width, target_height), Image.LANCZOS)

    # Create a blank canvas with the target size
    canvas = Image.new("RGB", target_size, color=(0, 0, 0))

    # Paste the resized image onto the canvas at the top-left
    canvas.paste(im, (0, 0))
    
    return canvas

In [ ]:
# Load updated master spreadsheet
meta = pd.read_csv(f"/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/master_spreadsheet/matched_su_mxa.csv", dtype=str)
gby = meta.groupby(["UM Accession", "Block"])["Stain"].agg(list).reset_index()
reg_cand = gby.loc[gby.apply(alignment_filter, axis=1)]

# Find existing annotations (match by SU)
annot_root = "/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/playgrounds/pixel_alignment/sections_annot2"
existing_block_meta_paths = glob(opj(annot_root, "meta", "*_sections_annot_meta.csv"))  
existing_block_metas = pd.concat([pd.read_csv(egmp, dtype=str) for egmp in existing_block_meta_paths
                                 if os.path.exists(opj(annot_root, "block_annot", egmp.split("/")[-1].removesuffix("_sections_annot_meta.csv")+".tiff"))
                                 ])
existing_blocks = existing_block_metas[["UM Accession", "Block"]].drop_duplicates()

# Make sure previously annotated blocks haven't changed
need_to_redo = []
newest_gb = meta.groupby(["UM Accession", "Block"])
newest_gb_blocks = {k for k, g in newest_gb}
for k, g in tqdm(existing_block_metas.groupby(["UM Accession", "Block"])):
    if k not in newest_gb_blocks:
        need_to_redo.append(k)
        print(need_to_redo)
        print(existing_cb.drop(["tile", "comment", "Unnamed: 0"], axis=1))
        print("---")
        print("NONE")
        print("===")
    else:
        existing_cb = g #g[~(g["comment"]=="RM")]
        existing_cb = existing_cb.reset_index(drop=True)
        newest_cb = meta.iloc[newest_gb.groups[k]].reset_index(drop=True)
        newest_cb_idxs = newest_cb[newest_cb["Stain"]=="H&E"].index.tolist() + newest_cb[~ (newest_cb["Stain"]=="H&E")].index.tolist()
        newest_cb = newest_cb.iloc[newest_cb_idxs].reset_index(drop=True)

        if not existing_cb.drop(["tile", "comment", "Unnamed: 0"], axis=1).equals(newest_cb.drop(["tile", "comment"], axis=1)):
            need_to_redo.append(k)
            print(existing_cb.drop(["tile", "comment", "Unnamed: 0"], axis=1))
            print("---")
            print(newest_cb.drop(["tile", "comment"], axis=1))
            print("===")

need_to_redo = pd.DataFrame(need_to_redo, columns=["UM Accession", "Block"])

# Filter reg candidates
reg_cand = reg_cand.merge(existing_blocks, on=["UM Accession", "Block"], how='left', indicator=True)
reg_cand = reg_cand[reg_cand['_merge'] == 'left_only'].drop(columns=['_merge'])

# Exclude any partally scanned, incomplete cases
#reg_cand = reg_cand.loc[~ reg_cand["UM Accession"].isin(["SU-22-85671", "SU-22-37460", "SU-22-39587"]), :]

In [ ]:
need_to_redo = pd.DataFrame([("SU-15-62441", "B3"), ("SU-15-65869", "UNK"), ("SU-19-49105", "B1"), ("SU-19-90632", "C1")], columns=["UM Accession", "Block"]) 

In [ ]:
need_to_redo

In [ ]:
svs_root = "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/svs"

for _, rc in tqdm(need_to_redo.iterrows(), total=len(need_to_redo)):
    curr_block_str = ".".join((rc["UM Accession"], rc["Block"]))

    im_b = []
    curr_block = meta[(meta["UM Accession"]==rc["UM Accession"]) & (meta["Block"]==rc["Block"])]
    cb_idxs = curr_block[curr_block["Stain"]=="H&E"].index.tolist() + curr_block[~ (curr_block["Stain"]=="H&E")].index.tolist()

    for he_i, he_s in curr_block.loc[cb_idxs].iterrows():
        slide = openslide.OpenSlide(opj(svs_root, he_s["path"]))
        im_b.append(resize_and_pad_to_size_one_im(Image.fromarray(get_thumbnail(slide)), (768,768)))

    curr_block.loc[cb_idxs].to_csv(opj("sections_annot2/reannot", f"{curr_block_str}_sections_annot_meta.csv"), index=False)
    im_b[0].save(opj(annot_root, "reannot", f"{curr_block_str}.tiff"), save_all=True, append_images=im_b[1:])


In [ ]:
# Add group numbers
existing_group_meta_paths = glob(opj(annot_root, "meta", "group*_meta.csv"))
start_group = max([int(os.path.basename(x).removeprefix("group").removesuffix("_meta.csv")) for x in existing_group_meta_paths]) + 1
num_elts = reg_cand['Stain'].apply(len)
reg_cand["group"] = (num_elts.cumsum() // 50).astype(int) + start_group

In [ ]:
print(f'New reg candidate blocks / SUs: {len(reg_cand)} / {len(set(reg_cand["UM Accession"]))}')
reg_cand

In [ ]:
svs_root = "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/svs"
for i, dfg in tqdm(reg_cand.groupby("group")):
    meta_g = []
    im_g = []

    for _, rc in dfg.iterrows():
        curr_block_str = ".".join((rc["UM Accession"], rc["Block"]))

        try:
            curr_block = meta[(meta["UM Accession"]==rc["UM Accession"]) & (meta["Block"]==rc["Block"])]
            cb_idxs = curr_block[curr_block["Stain"]=="H&E"].index.tolist() + curr_block[~ (curr_block["Stain"]=="H&E")].index.tolist()

            for he_i, he_s in curr_block.loc[cb_idxs].iterrows():
                slide = openslide.OpenSlide(opj(svs_root, he_s["path"]))
                im_g.append(resize_and_pad_to_size_one_im(Image.fromarray(get_thumbnail(slide)), (768,768)))

            meta_g.append(curr_block.loc[cb_idxs])
            curr_block.loc[cb_idxs].to_csv(opj("sections_annot2/meta", f"{curr_block_str}_sections_annot_meta.csv"), index=False)
        except:
            print(f"FAILED - {curr_block_str}")
  
    im_g[0].save(opj(annot_root, "images", f"group{i}.tiff"), save_all=True, append_images=im_g[1:])
    pd.concat(meta_g).to_csv(opj(annot_root, "meta", f"group{i}_meta.csv"), index=False)

In [ ]:
trdb_meta[to_keep].to_csv(opj(annot_root, "reannot", f"{trdb}_sections_annot_meta.csv"), index=False)

# Code to remove RM rows in the annot meta

In [ ]:
import tifffile

In [ ]:
annot_root = "/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/playgrounds/pixel_alignment/sections_annot2"
rm_blocks = ["SU-16-69491.D1", "SU-15-39138.B2"]


In [ ]:
for rmb in rm_blocks:
    meta_rmb = pd.read_csv(opj(annot_root, "meta", f"{rmb}_sections_annot_meta.csv"))
    to_rm = meta_rmb["comment"] == "RM"

    annot_rmb = tifffile.imread(opj(annot_root, "block_annot", f"{rmb}.tiff"))

    rmed_annot = [Image.fromarray(i) for i in annot_rmb[~to_rm, ...]]
    rmed_meta = meta_rmb[~to_rm]

    rmed_annot[0].save(opj(annot_root, "block_annot", f"{rmb}.tiff"), save_all=True, append_images=rmed_annot[1:])
    rmed_meta.to_csv(opj(annot_root, "meta", f"{rmb}_sections_annot_meta.csv"), index=False)